In [7]:
from pathlib import Path
import numpy as np
import pandas as pd
from catboost import CatBoostClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

In [8]:
data_directory = Path(__name__).resolve().parent.parent / 'data'
train_data = pd.read_csv(data_directory / 'train_data.csv')

X_train = train_data.drop('Churn', axis=1)
y_train = train_data['Churn']

In [9]:
models = {
    'logreg': LogisticRegression(class_weight='balanced', random_state=42),
    'decision_tree': DecisionTreeClassifier(random_state=42, class_weight='balanced'),
    'random_forest': RandomForestClassifier(random_state=42, class_weight='balanced'),
    'svc': SVC(random_state=42, class_weight='balanced', probability=True),
    'gradient_boost': GradientBoostingClassifier(random_state=42, verbose=0),
    'catboost': CatBoostClassifier(random_state=42, verbose=False, auto_class_weights='Balanced')
}

scores = {}

for name, model in models.items():
    strat_kfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    score = cross_validate(model,
                           X_train, y_train,
                           cv=strat_kfold,
                           scoring=['precision', 'recall','roc_auc'],
                           n_jobs=-1)

    metrics = [score['test_precision'].mean(), score['test_recall'].mean(), score['test_roc_auc'].mean()]
    scores[name] = metrics

In [10]:
pd.DataFrame(scores, index=['precision', 'recall', 'roc_auc'])

,logreg,decision_tree,random_forest,svc,gradient_boost,catboost
precision,0.515793,0.500644,0.622365,0.518057,0.651293,0.544978
recall,0.796680,0.499990,0.455768,0.773861,0.517100,0.730341
roc_auc,0.843855,0.659972,0.821862,0.823882,0.845617,0.840492
